In [3]:

from pathlib import Path

D = Path("/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation")
V = D / "verifiers" / "invsqrt"

print("Verifier files:")
for f in sorted(V.rglob("*")):
    if f.is_file():
        print(f.relative_to(D))

print("\nBaseline circuit:")
print(D / "sample_submission" / "invsqrt.kmb")


Verifier files:
verifiers/invsqrt/__init__.py
verifiers/invsqrt/nr_reference_circuit.py
verifiers/invsqrt/verify_submission_invsqrt.py

Baseline circuit:
/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation/sample_submission/invsqrt.kmb



# 00 — Baseline Verification

Official baseline verification for Google's Verifiable Quantum Circuit Compilation competition.

- Dataset: Official Kaggle competition input
- Circuits: Unmodified sample submissions
- Purpose: Establish verified baseline costs and correctness
- Status: Not yet verified


In [4]:

from pathlib import Path
import os
import sys

DATA = Path(
    "/kaggle/input/competitions/"
    "google-verifiable-quantum-circuit-compilation"
)

SAMPLES = DATA / "sample_submission"
VERIFIERS = DATA / "verifiers"

assert DATA.exists(), "Competition dataset not found"
assert (SAMPLES / "invsqrt.kmb").exists()
assert (VERIFIERS / "invsqrt" / "verify_submission_invsqrt.py").exists()

print("Environment paths verified.")
print("Dataset:", DATA)


Environment paths verified.
Dataset: /kaggle/input/competitions/google-verifiable-quantum-circuit-compilation



## 01 — Official Verifier Smoke Test

Run the unmodified `invsqrt` baseline against a small subset of inputs.

This confirms that KickMix loads and the official verifier executes correctly. A successful smoke test is **not** proof that the circuit passes the full benchmark.


In [8]:

import subprocess
import time

circuit = SAMPLES / "invsqrt.kmb"
verifier = VERIFIERS / "invsqrt" / "verify_submission_invsqrt.py"

# Ensure Python loads the official competition KickMix library.
env = dict(os.environ)
env["PYTHONPATH"] = str(DATA / "kickmix")

cmd = [
    sys.executable,
    str(verifier),
    str(circuit),
    "--max-x", "1024",
    "--shots-per-x", "1",
    "--batch-size", "64",
    "--max-workers", "2",
]

start = time.perf_counter()

result = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
)

elapsed = time.perf_counter() - start

print("Exit code:", result.returncode)
print("Elapsed:", round(elapsed, 2), "seconds")
print("STDOUT:\n", result.stdout[-6000:])
print("STDERR:\n", result.stderr[-3000:])


Exit code: 0
Elapsed: 0.24 seconds
STDOUT:
 {
  "status": "PASS",
  "score": 2068518.259331096,
  "toffoli_equivalent": 7955.839458965754,
  "toffoli_count": 7735.0,
  "max_toffoli_count": 7735,
  "t_count": 1.0,
  "rotation_count": 37.0,
  "rotation_eps": 1.070829675108564e-10,
  "num_qubits": 260,
  "reaction_depth": 4005,
  "num_ops": 62160,
  "verified_shots": 1024,
  "worst_error_over_tolerance": 0.42871093750805755,
  "worst_x": 729,
  "circuit_sha256": "e2bf8eb64350a4460a271fc7f1d591ef125b1e276377e9a0f64b96ab3ca321e1"
}

STDERR:
 



## 02 — Full Official invsqrt Baseline Verification

Run the original baseline circuit using all official verifier defaults.

- Inputs: 1 through 3,133,452
- Shots per input: 64
- Total shots: 200,540,928
- Circuit: Unmodified sample submission

A PASS here establishes correctness under the full official invsqrt verifier. This is separate from submitting to the Kaggle leaderboard.


In [9]:

import json
import subprocess
import time

full_cmd = [
    sys.executable,
    str(VERIFIERS / "invsqrt" / "verify_submission_invsqrt.py"),
    str(SAMPLES / "invsqrt.kmb"),
]

start = time.perf_counter()

with open("/kaggle/working/invsqrt_full_stdout.txt", "w") as out, \
     open("/kaggle/working/invsqrt_full_stderr.txt", "w") as err:
    process = subprocess.run(
        full_cmd,
        env=env,
        stdout=out,
        stderr=err,
    )

elapsed = time.perf_counter() - start

print("Exit code:", process.returncode)
print("Runtime (minutes):", round(elapsed / 60, 2))

stdout = Path("/kaggle/working/invsqrt_full_stdout.txt").read_text()
stderr = Path("/kaggle/working/invsqrt_full_stderr.txt").read_text()

print("Verifier output:")
print(stdout[-6000:])
if stderr:
    print("Errors:", stderr[-3000:])

if process.returncode == 0:
    try:
        result_data = json.loads(stdout)
        Path("/kaggle/working/invsqrt_full_results.json").write_text(
            json.dumps(result_data, indent=2)
        )
        print("Results saved successfully.")
    except json.JSONDecodeError:
        print("Output saved as text; JSON parsing unsuccessful.")


Exit code: 0
Runtime (minutes): 7.58
Verifier output:
{
  "status": "PASS",
  "score": 2076774.233715282,
  "toffoli_equivalent": 7987.593206597238,
  "toffoli_count": 7735.0,
  "max_toffoli_count": 7735,
  "t_count": 1.0,
  "rotation_count": 37.0,
  "rotation_eps": 1.5289254939242122e-12,
  "num_qubits": 260,
  "reaction_depth": 4005,
  "num_ops": 62160,
  "verified_shots": 200540928,
  "worst_error_over_tolerance": 0.4769116097939472,
  "worst_x": 2588411,
  "circuit_sha256": "e2bf8eb64350a4460a271fc7f1d591ef125b1e276377e9a0f64b96ab3ca321e1"
}

Results saved successfully.



## 03 — DQI Baseline Verification

Validate the unmodified DQI baseline using the official verifier.

The initial 256-shot test executed successfully with zero errors, but failed the strict statistical confidence requirement due to insufficient samples.

This section starts with 2,048 shots before the full 102,400-shot benchmark.


In [14]:

import subprocess
import time

dqi_cmd = [
    sys.executable,
    str(VERIFIERS / "dqi" / "verify_submission_dqi.py"),
    str(SAMPLES / "dqi.kmb"),
    "--num-shots", "2048",
    "--batch-size", "256",
    "--max-workers", "1",
]

start = time.perf_counter()

dqi_result = subprocess.run(
    dqi_cmd,
    env=env,
    capture_output=True,
    text=True,
)

print("Exit code:", dqi_result.returncode)
print("Runtime (seconds):", round(time.perf_counter() - start, 2))
print("STDOUT:\n", dqi_result.stdout[-8000:])
print("STDERR:\n", dqi_result.stderr[-4000:])


Exit code: 0
Runtime (seconds): 18.67
STDOUT:
 {
  "status": "PASS",
  "score": 154025541234591.47,
  "m": 4095,
  "n": 70,
  "degree": 12,
  "r": 2016,
  "verified_shots": 2048,
  "value_errors": 0,
  "phase_errors": 0,
  "empirical_error_rate": 0.0,
  "effective_error_rate": 0.04239671930142635,
  "log2_delta": -128.0,
  "repetition_factor": 7.507732187290039,
  "mean_error_weight": 31.046737331746634,
  "toffoli_count": 5725095.603515625,
  "max_toffoli_count": 5725461,
  "reaction_depth": 170757,
  "num_ops": 86051014,
  "shot_operation_counts": {
    "NEG": 1331566,
    "BIT_INVERT": 6634678642,
    "BIT_STORE0": 1411873368,
    "BIT_STORE1": 50318624,
    "X": 1696436288,
    "Z": 34546343,
    "R": 2513943696,
    "HMR": 2513943696,
    "CX": 110318596900,
    "CZ": 9666476956,
    "SWAP": 6418528512,
    "CSWAP": 2401517568,
    "CCX": 9323478228,
    "INFO_TAG": 6670594048,
    "POP_CONDITION": 86624256,
    "PUSH_CONDITION": 86624256
  },
  "num_qubits": 1893,
  "num_bits": 4


## 04 — Full Official DQI Baseline Verification

Verify the unmodified DQI sample circuit with the official benchmark settings.

- Shots: 102,400
- Batch size: 256
- Workers: 1 (memory-conscious configuration)
- Circuit: Original sample submission

Record the full statistical confidence bound, repetition factor, gate counts, and final DQI baseline cost.


In [15]:

import json
import subprocess
import time

dqi_full_cmd = [
    sys.executable,
    str(VERIFIERS / "dqi" / "verify_submission_dqi.py"),
    str(SAMPLES / "dqi.kmb"),
    "--num-shots", "102400",
    "--batch-size", "256",
    "--max-workers", "1",
]

start = time.perf_counter()

with open("/kaggle/working/dqi_full_stdout.txt", "w") as out, \
     open("/kaggle/working/dqi_full_stderr.txt", "w") as err:
    process = subprocess.run(
        dqi_full_cmd,
        env=env,
        stdout=out,
        stderr=err,
    )

elapsed = time.perf_counter() - start

print("Exit code:", process.returncode)
print("Runtime (minutes):", round(elapsed / 60, 2))

stdout = Path("/kaggle/working/dqi_full_stdout.txt").read_text()
stderr = Path("/kaggle/working/dqi_full_stderr.txt").read_text()

if process.returncode == 0:
    result_data = json.loads(stdout)
    Path("/kaggle/working/dqi_full_results.json").write_text(
        json.dumps(result_data, indent=2)
    )

    for key in [
        "status",
        "score",
        "verified_shots",
        "value_errors",
        "phase_errors",
        "effective_error_rate",
        "repetition_factor",
        "toffoli_count",
        "num_qubits",
        "circuit_sha256",
    ]:
        print(f"{key}: {result_data.get(key)}")

    print("Full results saved.")
else:
    print("STDOUT:", stdout[-6000:])
    print("STDERR:", stderr[-4000:])


Exit code: 0
Runtime (minutes): 5.83
status: PASS
score: 24439463583672.93
verified_shots: 102400
value_errors: 0
phase_errors: 0
effective_error_rate: 0.0008660587301657926
repetition_factor: 1.1912604648115832
toffoli_count: 5725108.382109375
num_qubits: 1893
circuit_sha256: 959471d05a26373b4f14ca0cae2d8639ea25f18170d9f6532139aa51ccc04995
Full results saved.



## 05 — Full Official DFTHCSOS Baseline Verification

Verify the unmodified DFTHCSOS sample submission using all official verifier defaults.

- System: FeMoCo76o113e
- Circuits: 10 ANGLE_k circuits
- Factorisation: submission.npz
- Shots per address: 64 (default)
- Expected reference cost: 3,355,599,588.3155365

This establishes the third benchmark's baseline cost and checks the numerical and quantum-circuit acceptance requirements.


In [18]:

import json
import subprocess
import time
from pathlib import Path

dfth_verifier = VERIFIERS / "dfthcsos" / "verify.py"
dfth_sample = SAMPLES / "dfthcsos"

cmd = [
    sys.executable,
    str(dfth_verifier),
    str(dfth_sample),
]

start = time.perf_counter()

result = subprocess.run(
    cmd,
    cwd=str(VERIFIERS),
    env=env,
    capture_output=True,
    text=True,
)

elapsed = time.perf_counter() - start

Path("/kaggle/working/dfthcsos_full_stdout.txt").write_text(result.stdout)
Path("/kaggle/working/dfthcsos_full_stderr.txt").write_text(result.stderr)

print("Exit code:", result.returncode)
print("Runtime (seconds):", round(elapsed, 2))

if result.returncode == 0:
    data = json.loads(result.stdout)

    Path("/kaggle/working/dfthcsos_full_results.json").write_text(
        json.dumps(data, indent=2)
    )

    for key in [
        "status",
        "score",
        "lambda_eff",
        "toffoli_count",
        "num_qubits",
        "verified_shots",
        "submission_sha256",
    ]:
        print(f"{key}: {data.get(key)}")

    print("\nFull results saved.")
else:
    print("STDOUT:\n", result.stdout[-6000:])
    print("STDERR:\n", result.stderr[-4000:])


Exit code: 0
Runtime (seconds): 53.78
status: PASS
score: 3355599588.3155375
lambda_eff: 49.301369250574396
toffoli_count: 93237.0
num_qubits: 730
verified_shots: 254464
submission_sha256: 276d464ff14da828b6b33e8487d99975494cbdf0c3cb2bd37dd3c3ddc4002695

Full results saved.
